# 14 — Where Do LLMs Store Knowledge?

**Description:** Build a toy associative memory in an MLP, where feature detectors recognize entities and write associated facts into the residual stream.
**Level:** Beginner
**Tags:** Language Models, Knowledge, Associative Memory, MLP, Interpretability

Transformer knowledge is distributed across many parameters and computations; there is no single universal “fact database” inside an LLM. Still, MLPs can implement a useful pattern:

$$\text{detect a feature} \rightarrow \text{activate} \rightarrow \text{write an associated feature}$$

This notebook builds a transparent toy memory for associations such as `Michael Jordan → basketball`. It is an intuition pump, not a literal map of where every fact lives in a real model.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=3, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Separate entity and attribute directions

We construct an 8D residual space. The first four coordinates encode entity features; the last four encode attributes that the memory can write.

In [ ]:
entities = ["Michael Jordan", "Marie Curie", "Paris", "Mozart"]
attributes = ["basketball", "physics", "France", "music"]
d_model = 8

entity_vectors = np.eye(d_model)[:4]
attribute_vectors = np.eye(d_model)[4:]

for entity, vector in zip(entities, entity_vectors):
    print(f"{entity:>14}: {vector}")

These one-hot directions make the mechanism obvious. Real representations are dense, learned, contextual, and distributed.

## 2. Detection: keys recognize entities

Treat each entity direction as a detector. The score matrix compares every input with every stored key.

In [ ]:
keys = entity_vectors.copy()
inputs = np.stack([
    entity_vectors[0],
    entity_vectors[1],
    0.8 * entity_vectors[2] + 0.2 * entity_vectors[3],
])
input_names = ["Michael Jordan", "Marie Curie", "mostly Paris"]
scores = inputs @ keys.T

print("scores shape:", scores.shape)
print(scores)

## 3. Nonlinearity makes selective memory activations

A thresholded ReLU turns similarity into a sparse activation. Only strong matches retrieve a memory.

In [ ]:
threshold = 0.6
memory_activations = np.maximum(0.0, scores - threshold)

fig, ax = plt.subplots(figsize=(8, 3.5))
image = ax.imshow(memory_activations, cmap="magma", aspect="auto")
ax.set_xticks(range(len(entities)), entities, rotation=25, ha="right")
ax.set_yticks(range(len(input_names)), input_names)
ax.set(xlabel="memory detector", ylabel="input", title="Sparse entity-memory activations")
for i in range(len(input_names)):
    for j in range(len(entities)):
        ax.text(j, i, f"{memory_activations[i,j]:.2f}", ha="center", va="center")
fig.colorbar(image, ax=ax, label="activation")
plt.show()

## 4. Writing: values contain associated attributes

Pair each detector with an attribute direction. Matrix multiplication combines the write vectors according to the detector activations.

In [ ]:
writes = attribute_vectors.copy()
retrieved = memory_activations @ writes
updated = inputs + retrieved

for name, activation, write in zip(input_names, memory_activations, retrieved):
    active = [(attributes[i], round(value, 3)) for i, value in enumerate(activation) if value > 0]
    print(f"{name:>14} retrieves {active}; write={write}")

For `Michael Jordan`, the first detector activates and writes the `basketball` direction. Detection and writing use different vectors, just like the $W_{up}$ columns and $W_{down}$ rows in an MLP.

## 5. Package the toy associative memory

In [ ]:
def associative_memory(x, keys, writes, threshold=0.6):
    activations = np.maximum(0.0, x @ keys.T - threshold)
    update = activations @ writes
    return x + update, activations, update

result, activations, update = associative_memory(inputs, keys, writes)
assert np.allclose(result, updated)
print("input -> activations -> update -> result")
print(inputs.shape, activations.shape, update.shape, result.shape)

## 6. Generalization depends on feature similarity

A noisy entity representation can still trigger the same memory if it remains aligned with the detector.

In [ ]:
rng = np.random.default_rng(14)
clean = entity_vectors[0]
noise_levels = np.linspace(0, 1.5, 30)
activation_levels = []
for level in noise_levels:
    noisy = clean + level * rng.normal(size=d_model)
    noisy /= np.linalg.norm(noisy)
    _, activation, _ = associative_memory(noisy[None, :], keys, writes)
    activation_levels.append(activation[0, 0])

plt.plot(noise_levels, activation_levels, "o-")
plt.axhline(0, color="gray", linewidth=0.8)
plt.xlabel("noise level")
plt.ylabel("Michael Jordan memory activation")
plt.title("Retrieval weakens as the entity feature becomes noisy")
plt.show()

## 7. An ambiguous mixture can retrieve several associations

Dense activations allow mixtures. This is powerful, but it also means retrieval can interfere when features overlap.

In [ ]:
mixture = 0.8 * entity_vectors[0] + 0.8 * entity_vectors[1]
result, activation, update = associative_memory(mixture[None, :], keys, writes)
print("activations:", activation[0])
print("attribute write:", update[0, 4:])

## 8. Attention versus MLP memory

Both mechanisms use detection and weighted writing, but their sources differ:

| Mechanism | Detects against | Writes from | Main role |
| --- | --- | --- | --- |
| Attention | other token positions in the current sequence | their value vectors | move contextual information |
| MLP | learned parameter directions | learned write directions | transform/retrieve parameterized associations |

This distinction is useful, but not absolute. Knowledge can involve embeddings, attention heads, MLPs, normalization, and interactions across many layers.

## 9. What this toy model leaves out

- Real facts are not usually represented by clean one-hot coordinates.
- A model may encode one association across many neurons and layers.
- One neuron may participate in many apparently unrelated features.
- Context controls which knowledge is relevant and how it affects logits.
- Memorized associations can be incomplete, conflicting, or wrong.

So “MLPs are key–value memories” is a productive lens, not a complete theory of model knowledge.

## 10. Challenges

1. Add `Ada Lovelace → computing` by increasing the model width.
2. Make two entity keys partially overlap and measure interference.
3. Replace ReLU retrieval with softmax. How does the behavior change?
4. Store two attributes for one entity.
5. Explain how context from attention could move a token closer to the right entity detector.

## Takeaways

- An MLP can behave like an associative memory: detect an input feature and write an associated output feature.
- $W_{up}$ supplies detection directions; $W_{down}$ supplies write directions.
- Feature similarity can support robust retrieval and can also cause interference.
- Attention mainly routes information from the current context, while MLP parameters can encode learned transformations and associations.
- Real model knowledge is distributed; the toy memory is a mechanism-level intuition, not a literal lookup table.
- Notebook 15 explains why features need not align one-to-one with neurons.